# Causal Forest trên Kaggle — notebook chạy được ngay

Notebook này hoàn tất **hạng mục còn thiếu duy nhất** của dự án: chạy `CausalForestDML`
ở quy mô thật. Laptop 15,19 GB không đủ (ngoại suy 50% cần khoảng 17,5 GB).

**Trước khi chạy — ba việc bắt buộc:**

1. Settings → Accelerator = **None (CPU)**. GPU không làm forest nhanh hơn vì
   `CausalForestDML` dùng CPU parallelism và system RAM.
2. Settings → Internet = **On** (cần cho `pip install` và `git clone`).
3. Add Input → attach dataset Criteo v2.1.

**Chạy tuần tự từng cell, không Run All.** Cell 5 yêu cầu restart kernel; Run All sẽ
bỏ qua bước đó và import EconML sẽ lỗi.

Runbook giải thích lý do từng bước: `docs/KAGGLE_RUNBOOK_COMPLETE.md`.


## Cell 1 — Đọc tài nguyên thật của session

Không giả định Kaggle luôn có cấu hình cố định. Ghi lại `RAM total`: toàn bộ stop rule
tính theo tỷ lệ trên con số này.


In [ ]:
import os, platform, shutil
import psutil

vm = psutil.virtual_memory()
disk = shutil.disk_usage('/kaggle/working')
print(f'python            {platform.python_version()}')
print(f'logical cpus      {os.cpu_count()}')
print(f'physical cpus     {psutil.cpu_count(logical=False)}')
print(f'RAM total         {vm.total / 2**30:.2f} GB')
print(f'RAM available     {vm.available / 2**30:.2f} GB')
print(f'working disk free {disk.free / 2**30:.2f} GB')
print()
for root, _, files in os.walk('/kaggle/input'):
    for name in files:
        p = os.path.join(root, name)
        print(f'{os.path.getsize(p) / 2**20:10.1f} MB  {p}')

if vm.total / 2**30 < 20:
    print()
    print('CANH BAO: RAM total duoi 20 GB. KHONG chay stage 50%.')
    print('Ket thuc o learning curve 20-30% va bao cao dung nhu vay.')


## Cell 2 — Xác minh checksum dữ liệu

Mất khoảng một phút, tránh việc chạy ba tiếng trên sai dataset.


In [ ]:
import glob, hashlib

EXPECTED = '2716e1bf0fd157a93b5bf86924d9088419dfbac2022c6cd90030220634f616dc'
matches = glob.glob('/kaggle/input/**/criteo-research-uplift-v2.1.csv.gz', recursive=True)
assert matches, 'Chua attach dataset Criteo v2.1 (Add Input o thanh ben phai)'
DATA_PATH = matches[0]

digest = hashlib.sha256()
with open(DATA_PATH, 'rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(chunk)
actual = digest.hexdigest()
print(DATA_PATH)
print('sha256', actual)
assert actual == EXPECTED, f'Checksum sai: {actual}'
print('checksum OK')


## Cell 3 — Đưa repository vào session

**Cách A** dùng khi repo đã public trên GitHub. **Cách B** dùng khi repo private hoặc
Internet = Off: nén repo (bỏ `data/`, `.venv/`, `output/`), upload thành Kaggle Dataset,
rồi đổi `USE_GITHUB = False` và điền tên dataset.


In [ ]:
import os, shutil, subprocess

USE_GITHUB = True
REPO_URL = 'https://github.com/ThanhDatVN/Causal-Uplift-for-Activation-and-Retention.git'
REPO_DATASET = '/kaggle/input/<ten-dataset-repo>'   # chi dung khi USE_GITHUB = False
REPO_DIR = '/kaggle/working/repo'

if not os.path.exists(REPO_DIR):
    if USE_GITHUB:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    else:
        shutil.copytree(REPO_DATASET, REPO_DIR, dirs_exist_ok=True)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
print(sorted(os.listdir('.')))

for required in ('scripts', 'src', 'configs'):
    assert os.path.isdir(required), f'Thieu thu muc {required} trong repo'
print('repo OK')


## Cell 4 — Cài dependency có ghim phiên bản

Đây là chỗ hỏng thường gặp nhất. `econml==0.16.0` yêu cầu `scikit-learn>=1.0,<1.7` và
`shap>=0.38.1,<0.49.0`. Image Kaggle thường có scikit-learn mới hơn giới hạn đó.


In [ ]:
import subprocess, sys

packages = [
    'econml==0.16.0',
    'scikit-learn>=1.4,<1.7',   # rang buoc cung cua econml 0.16
    'shap>=0.38.1,<0.49.0',     # rang buoc cung cua econml 0.16
    'lightgbm>=4.5',
    'psutil',
]
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
print(result.stderr[-2000:])
print('exit code:', result.returncode)


## Cell 5 — RESTART KERNEL tại đây

> **Bắt buộc.** Run → Restart session, rồi chạy tiếp từ Cell 6.
>
> Không restart thì scikit-learn cũ vẫn nằm trong bộ nhớ, và `import econml` sẽ lỗi
> hoặc chạy sai phiên bản. Đây là lỗi số 1 trong danh mục lỗi của runbook.

Sau khi restart, mọi biến Python đều mất. Cell 6 dựng lại tất cả.


## Cell 6 — Sau restart: dựng lại biến và kiểm tra phiên bản


In [ ]:
import glob, os

REPO_DIR = '/kaggle/working/repo'
OUTPUT_ROOT = '/kaggle/working/output/causal_forest'
os.chdir(REPO_DIR)
DATA_PATH = glob.glob('/kaggle/input/**/criteo-research-uplift-v2.1.csv.gz', recursive=True)[0]
print('cwd      ', os.getcwd())
print('data     ', DATA_PATH)
print('output   ', OUTPUT_ROOT)
print()

import numpy, scipy, sklearn, lightgbm, econml, pandas
for module in (numpy, scipy, sklearn, lightgbm, econml, pandas):
    print(f'{module.__name__:14s} {module.__version__}')

from packaging.version import Version
assert Version(sklearn.__version__) < Version('1.7'), 'scikit-learn phai < 1.7 cho econml 0.16'
assert econml.__version__ == '0.16.0'

import inspect
from econml.dml import CausalForestDML
assert 'inference' in inspect.signature(CausalForestDML.__init__).parameters
print()
print('moi thu OK, san sang chay stage 20%')


## Cell 7 — Hàm chạy một stage

Gate tự kiểm tra: checksum, không cho nhảy stage, exit code, peak RSS dưới 75% RAM,
score hữu hạn, số dòng khớp holdout.


In [ ]:
import json, subprocess, sys


def stage_slug(frac):
    """Khop ham _slug trong kaggle_causal_forest_gate.py: 0.2 -> '0p2'."""
    return str(frac).replace('.', 'p')


def run_stage(frac):
    """Chay mot stage, in tom tat. Tra manifest neu pass, None neu fail."""
    print(f'=== stage {frac:.0%} bat dau ===', flush=True)
    result = subprocess.run(
        [sys.executable, 'scripts/kaggle_causal_forest_gate.py',
         '--data-path', DATA_PATH,
         '--frac', str(frac),
         '--output-root', OUTPUT_ROOT,
         '--max-ram-fraction', '0.75'],
        capture_output=True, text=True,
    )
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print('STDERR:', result.stderr[-3000:])
        print(f'stage {frac:.0%} FAIL, exit code {result.returncode}')
        return None

    path = f'{OUTPUT_ROOT}/preflight_{stage_slug(frac)}/gate_manifest.json'
    with open(path) as handle:
        manifest = json.load(handle)
    runtime = manifest['runtime']
    print()
    print('status               ', manifest['status'])
    print(f"peak RSS              {runtime['peak_process_tree_rss_gb']:.2f} GB")
    print(f"peak RAM fraction     {runtime['peak_process_tree_ram_fraction']:.3f}")
    print(f"wall time             {runtime['wall_seconds'] / 60:.1f} phut")
    print('may continue         ', manifest['stop_rule']['may_continue'])
    print()
    for nxt in (0.30, 0.50):
        if nxt > frac:
            rss = runtime['peak_process_tree_rss_gb'] * nxt / frac
            minutes = runtime['wall_seconds'] * nxt / frac / 60
            print(f'du bao {nxt:.0%}: RSS ~ {rss:.2f} GB, time ~ {minutes:.0f} phut')
    return manifest


## Cell 8 — Stage 20%

Ước tính 35–60 phút tuỳ CPU của session.


In [ ]:
m20 = run_stage(0.20)


## Cell 9 — Stage 30%

Chỉ chạy khi stage 20% `passed` **và** dự báo RSS còn dưới 75% RAM total.


In [ ]:
assert m20 is not None and m20['status'] == 'passed', 'Stage 20% chua pass'
m30 = run_stage(0.30)


## Cell 10 — Stage 50%

Đây là stage **duy nhất** so được trực tiếp với bảng release Sprint 1: ở
`frac=0.50, test_size=0.30, seed=42`, holdout trùng khít final test Sprint 1
(2.096.940 dòng, `Y` và `T` giống hệt từng phần tử — đã kiểm chứng).

Ước tính 2–3,5 giờ. Nếu thời gian còn lại của session không đủ, **dừng ở đây** và báo
cáo learning curve 20–30%. Đó là kết quả hợp lệ, không phải thất bại.


In [ ]:
assert m30 is not None and m30['status'] == 'passed', 'Stage 30% chua pass'
m50 = run_stage(0.50)


## Cell 11 — Đóng gói output để tải về

Tải **cả file zip**, không chỉ chép một con số. Toàn bộ manifest, log và score phải về
repo để audit lại được.


In [ ]:
import os, shutil

archive = shutil.make_archive('/kaggle/working/causal_forest_output', 'zip', OUTPUT_ROOT)
print(f'{archive}  {os.path.getsize(archive) / 2**20:.1f} MB')
print()
for root, _, files in os.walk(OUTPUT_ROOT):
    for name in sorted(files):
        p = os.path.join(root, name)
        print(f'{os.path.getsize(p) / 2**20:9.2f} MB  {p}')


## Bước tiếp theo — chạy ở local sau khi tải về

Giải nén vào `output/causal_forest/` rồi chấm điểm:

```powershell
.venv\Scripts\python.exe scripts\evaluate_causal_forest.py `
  --stage-dir output\causal_forest\preflight_0p5 --n-boot 500 --signal dr
```

Với stage 20% và 30%, đổi `--stage-dir` thành `preflight_0p2` / `preflight_0p3`. Script
tự phát hiện và sẽ in `[mode] standalone` kèm cảnh báo rằng holdout đó **không** so được
với bảng release.

**Ba điều không được viết vào báo cáo:**

1. So Qini stage 20% với `0,187886` của Response — hai tập test khác nhau.
2. "Gate pass nghĩa là model tốt" — gate chỉ kiểm tài nguyên và toàn vẹn artifact;
   `"quality_not_assessed": true` nằm ngay trong manifest.
3. "Causal Forest cho khoảng tin cậy cá nhân" — profile `kaggle-safe` đặt
   `inference=False`, không gọi `effect_interval()` được.
